# Unused models

Models moved out of the main fatigue modeling path. Canonical continuous regression is [`4 fatigue regression.ipynb`](4%20fatigue%20regression.ipynb) (2 tuned models: `linear_regression`, `ordinal_rf`).

## Why unused?

### GEE models (`gee_gaussian`, `gee_ordinal`)

The classification notebook targets **prediction accuracy** (held-out test MAE). GEE is designed for **longitudinal inference**: coefficient estimates with **statistically valid standard errors and p-values** after accounting for within-cluster correlation (repeated daily rows within each participant-interval). That is valuable for interpretability and hypothesis testing, not for ranking predictors by test error.

On this dataset both GEE models land mid-pack (~1.27 test MAE) versus stronger tree and history-feature models (~0.9–1.2 MAE). They stay here for reference and optional re-runs.

### Continuous regression (`elasticnet_regression`, `svr_regression`, `catboost_regressor`)

Removed from notebook 4 after Optuna **MAE** tuning on the 20-feature pipeline:

- **ElasticNet:** L1 sparsity drove all feature weights to zero; test predictions collapsed to the train/val mean fatigue (~2.46) for every row.
- **SVR (RBF):** tuned `C` / `gamma` produced a nearly flat kernel; test predictions cluster around one value (test MAE ≈ `global_mode` baseline).
- **CatBoost regressor:** second tree ensemble on the same split; test/CV MAE nearly matched `ordinal_rf` (~1.31 test MAE). Removed from notebook 4 to avoid redundant tree reporting.

Ridge and tree regressors on the same split still spread predictions and add value. These models are kept here for optional re-runs, not the main §3–§6 comparison.

Same participant-level split, GroupKFold CV, and Optuna tuning as notebook 4.


In [ ]:
%pip install -q -r ../../requirements.txt


In [ ]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

from modeling.config import (
    DATA_PATH,
    ELASTICNET_OPTUNA_TRIALS,
    GEE_OPTUNA_TRIALS,
    N_CV_FOLDS,
    OPTUNA_TRIALS,
    SVR_OPTUNA_TRIALS,
)
from modeling.data import load_fatigue_data, prepare_splits, split_summary_table
from modeling.registry import ORDINAL_MODELS
from modeling.runner import tune_and_benchmark_model, print_tune_summary
from modeling.summaries import collect_summaries


## Load data and split


In [ ]:
df = load_fatigue_data('../../' + DATA_PATH)
bundle = prepare_splits(df)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle))
print('Test participant ids:', sorted(bundle.test_ids))


In [ ]:
ordinal_results = []
ordinal_best_params = {}


## Continuous regression (removed from notebook 4)


### `elasticnet_regression`

MSE with combined L1 (Lasso) and L2 (Ridge) penalties on scaled/OHE daily features. **50 Optuna trials** (`alpha` log [1e-3, 10], `l1_ratio` [0, 1]).

**Why unused:** Under MAE CV, L1 sparsity zeroed feature weights; predictions collapsed to train/val mean fatigue for all test rows.


In [ ]:
_name = 'elasticnet_regression'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=ELASTICNET_OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print_tune_summary(_name, _result, _params)


### `svr_regression`

RBF-kernel SVR with epsilon-insensitive loss on scaled/OHE daily features. **50 Optuna trials** (`C` log [0.01, 10], `epsilon` log [1e-3, 1], `gamma` scale/auto/float).

**Why unused:** Tuned `C` and `gamma` produced a nearly flat RBF; test MAE matched the `global_mode` baseline with almost no spread in predictions.


In [ ]:
_name = 'svr_regression'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=SVR_OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print_tune_summary(_name, _result, _params)


### `catboost_regressor`

Gradient-boosted trees on scaled/OHE daily features. **30 Optuna trials** (`iterations`, `depth`, `learning_rate`, `l2_leaf_reg` via `_catboost_search_space`).

**Why unused:** Near-identical test/CV MAE to `ordinal_rf` after tuning on this split; redundant second tree model for main §3–§6 reporting in notebook 4.


In [ ]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print_tune_summary(_name, _result, _params)


## GEE models


### `gee_gaussian` (ordinal regression)

**What it does:** Fits a **Gaussian GEE** on `fatigue_num`, treating the ordinal score as continuous. Uses **autoregressive working correlation** within each cluster (participant × `study_interval`, rows sorted by `day_in_study`). Includes `study_interval` as a covariate plus the 17 base daily features. Optuna tunes `maxiter`; the predicted mean is clipped to [0, 5].

**Why unused:** Inference-oriented (valid SEs/p-values under correlation). Not prioritized while we optimize prediction accuracy only.


In [ ]:
_name = 'gee_gaussian'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=GEE_OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'best params: {_params}')
print('GEE cluster = one participant-interval (id × study_interval); summary cluster sizes should be ~90 days per interval.')
print(_result['summary'])
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


### `gee_ordinal` (ordinal classification)

**What it does:** Fits **five cumulative-threshold Binomial GEE** models (P(y > k) for k = 0…4) with autoregressive working correlation (Exchangeable fallback when AR is unstable). Clustered by participant-interval; uses the 17 base daily features only (wave captured by clustering, not a `study_interval` covariate). Class probabilities from threshold survival are combined and clipped to [0, 5].

**Why unused:** Same inference vs accuracy tradeoff as `gee_gaussian`; kept for reference, not the main leaderboard.


In [ ]:
_name = 'gee_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=GEE_OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'best params: {_params}')
print('GEE cluster = one participant-interval (id × study_interval); summary cluster sizes should be ~90 days per interval.')
print(_result['summary'])
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


## Summary


In [ ]:
_, test_summary = collect_summaries(ordinal_results)
display(test_summary.sort_values('test_mae'))
